In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
import math
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from typing import Dict, Any, Tuple, List
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import ttest_rel
import copy
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
# Detect available physical GPU acceleration hardware
gpus = tf.config.list_physical_devices('GPU')

# Display the raw list of detected devices for verification
print(f"Physical Devices Detected: {gpus}")

# Verify GPU accessibility and configure memory management policies
if gpus:
    try:
        # Target the first GPU and enable dynamic memory allocation (Memory Growth).
        # This prevents TensorFlow from pre-allocating the entire VRAM, 
        # allowing for better resource management in shared environments.
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        tf.config.experimental.set_memory_growth(gpus[0], True)
        
        # Verify the logical configuration
        logical_gpus = tf.config.list_logical_devices('GPU')
        
        print(f"Configuration Status: {len(gpus)} Physical GPU(s), {len(logical_gpus)} Logical GPU(s).")
        print("Success: TensorFlow is correctly configured to utilize GPU acceleration.")
        
    except RuntimeError as e:
        # Memory growth must be set before GPUs have been initialized
        print(f"Runtime Error during GPU configuration: {e}")
else:
    print("Warning: No GPU detected. TensorFlow will default to CPU execution.")

In [ ]:
#--------------------------------------------------------------------------
# PARAMETER DEFINITIONS & UNITS:
# K          : Contact stiffness parameter (Hertzian) [N/m^(1.5)]
# delta      : Instantaneous indentation/penetration depth [m]
# delta_dot  : Instantaneous indentation velocity [m/s]
# e          : Coefficient of restitution (dimensionless, 0 < e <= 1)
# v_approach : Initial impact velocity (approach velocity) [m/s]
#--------------------------------------------------------------------------

def hunt_crossley_force(
    delta: float, 
    delta_dot: float, 
    K: float, 
    e: float, 
    v_approach: float, 
    **kwargs
) -> float:
    """
    Computes the normal contact force based on the Hunt-Crossley nonlinear 
    viscoelastic model.

    This model extends Hertzian contact theory by incorporating a dissipative 
    term to account for energy loss during impact, parametrized by the 
    coefficient of restitution.

    Governing Equation:
        F_N = K * delta^n + chi * delta^n * delta_dot

    Where 'chi' (hysteresis damping factor) is approximated as:
        chi = (3 * K * (1 - e)) / (2 * v_approach)

    Args:
        delta (float): Instantaneous indentation depth [m].
        delta_dot (float): Instantaneous indentation velocity [m/s].
        K (float): Hertzian stiffness parameter [N/m^1.5].
        e (float): Coefficient of restitution.
        v_approach (float): Initial approach velocity prior to impact [m/s].
        
    Returns:
        float: The calculated normal contact force [N]. 
               Returns 0.0 if there is no physical contact (delta <= 0).
    """
    
    # Physical constraint: No force is generated if bodies are not in contact
    if delta <= 0:
        return 0.0
        
    # Calculation of the hysteresis damping factor (chi)
    # This factor correlates the energy dissipation to the impact velocity
    chi = (3 * K * (1 - e)) / (2 * v_approach)

    # Total Normal Force calculation
    # Composition: Elastic Hertzian term + Dissipative Viscous term
    # F_N = K * δ^(1.5) + χ * δ^(1.5) * δ̇
    fn = (K * (delta**1.5)) + (chi * (delta**1.5) * delta_dot)
    
    return fn

### 2) Realizar experimentos simulados sobre o impacto de uma esfera em um plano sobre diferentes condições de velocidades de contato inicial.
 Resultados sobre diferentes modelagens de força de contato:

In [ ]:
# --- 1. Governing Equation of Motion (Newton's Second Law) ---
def impact_dynamics_model(t, state, force_law_func, params):
    """
    Differential equation governing the 1D impact dynamics.
    
    State Vector S:
        S[0]: delta (indentation depth) [m]
        S[1]: delta_dot (indentation velocity) [m/s]
        
    Returns:
        [velocity, acceleration]
    """
    delta, delta_dot = state
    mass = params['m']
    
    # Calculate Normal Force based on the specific constitutive law (e.g., Hunt-Crossley)
    force_magnitude = force_law_func(delta, delta_dot, **params)
    
    # Equation of Motion: a = -F / m
    # (Negative sign indicates the force opposes the penetration)
    acceleration = -force_magnitude / mass
    
    return [delta_dot, acceleration]

# --- 2. Simulation Setup & Experimental Parameters ---

# Reference Data: "Validation of Nonlinear Viscoelastic Contact Force Models for Low Speed Impact"
# "For the experiments presented in this manuscript, two materials were employed [...], 
# giving measured coefficients of restitution in the ranges 0.95–0.97 and 0.76–0.92, respectively."

restitution_coefficients = np.linspace(0.76, 0.97, 4) # Dimensionless
approach_velocities = [0.1, 0.15, 0.3, 0.4, 0.5]      # Initial Impact Velocities [m/s]

# Base Physical Parameters
sim_params_base = {
    'm': 0.54,          # Mass [kg]
    'v_approach': None, # Placeholder: updated iteratively [m/s]
    'K': 2.4614e8,      # Hertzian Stiffness Parameter [N/m^1.5]
    'Kl': 3e8,          # Linear (Hookean) Stiffness [N/m]
    'e': None,          # Placeholder: updated iteratively (Coefficient of Restitution)
    'alpha': 0.2,       # Parameter for Tsuji model
    'm_eff': 1.0,       # Effective mass (assumed equal to mass for Anagnostopoulos)
}

# List of Constitutive Models to Evaluate
models_to_simulate = [
    #{'name': 'Anagnostopoulos', 'func': anagnostopoulos_force},
    {'name': 'Hunt_Crossley', 'func': hunt_crossley_force},
    #{'name': 'Lankarni_Nikravesh', 'func': lankarni_nikravesh_force},
    #{'name': 'Flores_et_al', 'func': flores_et_al_force},
    #{'name': 'Tsuji_et_al', 'func': tsuji_et_al_force},
]

simulation_results = {}

print("--- Initiating Parameter Sweep Simulation ---")

# --- 3. Main Simulation Loop ---
for v_app in approach_velocities:
    for coef_e in restitution_coefficients:
        print(f"\nExecuting simulation run for: v_approach = {v_app} m/s | e = {coef_e:.3f}")
        
        # Update parameters for the current iteration
        current_run_params = sim_params_base.copy()
        current_run_params['v_approach'] = v_app
        current_run_params['e'] = coef_e
        
        # Initial Conditions: [Indentation = 0, Initial Velocity = v_app]
        initial_state = [0.0, v_app]
        
        for model in models_to_simulate:
            model_name = model['name']
            force_func = model['func']
            print(f"   > Model: {model_name}")

            # Temporal Settings
            t_span_max = 0.1 # Maximum duration [s]
            t_span = (0, t_span_max)
            # High resolution for smooth derivatives later
            t_eval = np.linspace(0, t_span_max, int(1e6)) 

            # Event Definition: Termination Condition
            # Stop simulation when indentation (S[0]) returns to 0 (separation)
            def end_contact_event(t, S, *args): return S[0]
            end_contact_event.terminal = True
            end_contact_event.direction = -1 # Triggers only when going from positive to negative
            
            # Solve ODE
            sol = solve_ivp(
                fun=impact_dynamics_model, 
                t_span=t_span, 
                y0=initial_state, 
                t_eval=t_eval,
                events=end_contact_event, 
                args=(force_func, current_run_params)
            )
            
            # Check if simulation terminated correctly via event
            if sol.status != 1: 
                print(f"      Warning: Simulation for {model_name} did not trigger end-contact event.")
                continue

            # Post-Processing: Compute Acceleration History
            # (acceleration is not stored directly in sol.y, so we recalculate it)
            acceleration = np.array([
                impact_dynamics_model(t, s, force_func, current_run_params)[1] 
                for t, s in zip(sol.t, sol.y.T)
            ])

            # Store Results
            # Key tuple: (Model Name, Approach Velocity, Restitution Coeff)
            result_key = (model_name, v_app, coef_e)
            
            simulation_results[result_key] = {
                'solution_object': sol,
                'position': sol.y[0],    # Indentation history
                'velocity': sol.y[1],    # Velocity history
                'acceleration': acceleration,
                'time': sol.t,
                'v_approach': v_app,
                'target_e': coef_e
            }
            
            # Verification: Calculate effective Coefficient of Restitution (COR)
            v_final = sol.y[1, -1] # Velocity at separation
            e_calculated = abs(v_final / v_app)
            
            print(f"      Result: Impact V: {v_app:.3f} m/s | Rebound V: {v_final:.3f} m/s | Calc. e: {e_calculated:.4f}")

print("\n--- Simulation Suite Completed Successfully ---")

In [ ]:
experiments_models = {}
models_names  = []

massa = sim_params_base['m']
for key, res in simulation_results.items():
    nome_modelo, v_app, coef_e = key

    # Gera dict com apenas as variáveis que eu posso mensurar experimentalmente
    experiments_models[key] = {
        'x': res['position'], # Array de posições,
        'v': res['velocity'], # Array de velocidades,
        'f_kN': (0.001)*(-1 * massa) * res['acceleration'], # Array de Forças
        'e': coef_e * np.ones(len(res['position'])), # Array de restituição de mesma dimensão dos arrays acima
        'v_approach': v_app * np.ones(len(res['position'])), # Array de restituição de mesma dimensão dos arrays acima
    }

    models_names.append(nome_modelo)


myset = set(models_names)

print("Done.")


In [ ]:
experiments_models.keys()

In [ ]:
def compile_reference_dataset(simulation_results: Dict[Tuple, Dict[str, Any]]) -> pd.DataFrame:
    """
    Aggregates individual simulation results into a unified Ground Truth DataFrame.
    
    This function standardizes the column nomenclature by appending the '_clean' 
    suffix, designating these signals as the noise-free reference targets for 
    subsequent supervised learning or denoising tasks.

    Args:
        simulation_results (Dict): A dictionary containing the raw simulation outputs.
            Structure: { (ModelName, v_approach, e): {'x': [...], 'v': [...], ...} }

    Returns:
        pd.DataFrame: A monolithic DataFrame containing all experimental time-series data,
                      with features renamed to indicate they represent the ground truth 
                      (e.g., 'x' becomes 'x_clean').
    """
    
    # Container to hold individual experiment DataFrames before concatenation
    experiment_frames = []
    
    # Schema mapping: Renaming raw output variables to "Clean/Reference" standard
    column_mapping = {
        'x': 'x_clean',                   # Displacement
        'v': 'v_clean',                   # Velocity
        'f_kN': 'f_kN_clean',             # Impact Force
        'e': 'e_clean',                   # Coefficient of Restitution
        'v_approach': 'v_approach_clean'  # Initial Approach Velocity
    }

    print("Compiling reference dataset from simulation results...")

    # Iterate through each simulation instance identified by its unique parameter tuple
    for experiment_id, time_series_data in simulation_results.items():
        
        try:
            # Convert the dictionary of arrays into a tabular DataFrame
            df_instance = pd.DataFrame(time_series_data)
            
            # Apply the standardized schema mapping
            # (Only renames columns present in the current instance)
            columns_to_rename = {k: v for k, v in column_mapping.items() if k in df_instance.columns}
            df_instance = df_instance.rename(columns=columns_to_rename)
            
            # Metadata Injection: Tag rows with the source experiment ID for traceability
            df_instance['experiment_id'] = str(experiment_id)
            
            # Append processed instance to the batch list
            experiment_frames.append(df_instance)
            
        except ValueError as e:
            # Captures common errors, such as array length mismatch within the dictionary
            print(f"Warning: Failed to process experiment {experiment_id}.")
            print(f"Diagnostic info: {e}")

    # Return an empty structure if no data was successfully processed
    if not experiment_frames:
        print("No valid data frames were generated.")
        return pd.DataFrame()

    # Concatenate all instances into a single memory-contiguous DataFrame
    df_ground_truth = pd.concat(experiment_frames, ignore_index=True)
    
    print(f"Successfully compiled Ground Truth dataset with {len(df_ground_truth)} samples.")
    
    return df_ground_truth

# Execution
df_clean = compile_reference_dataset(experiments_models)

In [ ]:
df_clean

In [ ]:
def add_gaussian_noise(
    models_data: Dict[Tuple, Dict[str, Any]], 
    noise_keys: List[str], 
    noise_level_percent: float
) -> Dict[Tuple, Dict[str, Any]]:
    """
    Adds Gaussian noise to specified signals within a nested dictionary.

    This function creates a deep copy of the input dictionary and adds
    Gaussian noise (np.random.normal) to the array-like data of
    the specified 'noise_keys'. The scale of the noise is determined
    by the signal's standard deviation multiplied by the 'noise_level_percent'.

    Args:
        models_data: The input dictionary of dictionaries.
            Example: { ('Hunt_Crossley', 0.1, 0.76): {'x': [..], 'v': [..], ...} }
        noise_keys: A list of strings (inner keys) specifying which signals 
            to add noise to (e.g., ['x', 'v', 'f_kN']).
        noise_level_percent: The percentage of the signal's standard deviation
            to use as the scale (std dev) for the Gaussian noise (e.g., 10.0 for 10%).

    Returns:
        A new dictionary with the same structure as models_data, but with
        Gaussian noise added to the specified signals. The original
        dictionary remains unmodified.
    """
    
    # Create a deep copy to ensure the original data is not modified
    noisy_models_data = copy.deepcopy(models_data)
    
    # Convert the noise percentage to a decimal multiplier
    # (e.g., 10.0 -> 0.10)
    noise_multiplier = noise_level_percent / 100.0

    # Iterate through each experiment in the top-level dictionary
    for experiment_key, signals_dict in noisy_models_data.items():
        
        # Iterate through each signal (e.g., 'x', 'v', 'e') in the nested dictionary
        for signal_key, signal_data in signals_dict.items():
            
            # Check if the current signal key is in the list of keys to add noise to
            if signal_key in noise_keys:
                
                # Ensure the data is a NumPy array for calculations
                signal_data_np = np.array(signal_data)
                
                # Calculate the standard deviation of the (clean) signal
                signal_std = np.std(signal_data_np)
                
                # Calculate the scale for the noise
                noise_scale = signal_std * noise_multiplier
                
                # Get the length of the signal array
                signal_length = len(signal_data_np)
                
                # Generate the Gaussian noise
                # loc=0.0 (mean), scale=noise_scale (std dev)
                noise = np.random.normal(loc=0.0, 
                                         scale=noise_scale, 
                                         size=signal_length)
                
                # Add the noise to the signal data
                noisy_signal = signal_data_np + noise
                
                # Update the value in our new (copied) dictionary
                signals_dict[signal_key] = noisy_signal
            
            # If signal_key is not in noise_keys, we do nothing,
            # as the deep copy already preserved the original value.

    return noisy_models_data

In [ ]:
keys_to_add_noise_to = ['x', 'v', 'f_kN']
noise_percentage = 1.0

noisy_data = add_gaussian_noise(
    models_data=experiments_models,
    noise_keys=keys_to_add_noise_to,
    noise_level_percent=noise_percentage
)


In [ ]:

def create_lagged_dataframe(
    models_data: Dict[Tuple, Dict[str, Any]], 
    num_points: int
) -> pd.DataFrame:
    """
    Converts a dictionary of experiments into a single Pandas DataFrame 
    with lagged (past) and leaded (future) features.

    Each experiment in the input dictionary is processed independently. For each
    signal (e.g., 'x'), it creates new columns for past and future values.
    For example, if num_points = 2, it will generate:
    'x_m_2', 'x_m_1', 'x', 'x_p_1', 'x_p_2'

    Boundary Conditions:
    For points at the beginning or end of a sequence where past or future
    data is not available, the value of the nearest boundary point (the
    first or last data point in that sequence) is used for padding.

    Args:
        models_data: The input dictionary of dictionaries.
            Example: { ('Hunt_Crossley', 0.1, 0.76): {'x': [..], 'v': [..], ...} }
        num_points: The number of past and future points to include as
            features (a non-negative integer).

    Returns:
        A single Pandas DataFrame containing the time-expanded features
        from all experiments. Includes 'experiment_key' and 
        'experiment_name' columns to identify the source of each row.
    """
    
    # A list to store the individual DataFrames for each experiment
    all_experiment_dfs = []

    # Iterate through each experiment in the top-level dictionary
    for experiment_key, signals_dict in models_data.items():
        try:
            # Convert the inner dictionary of signals into a DataFrame
            # This aligns all signals by their index (time step)
            experiment_df = pd.DataFrame(signals_dict)
            
            # Skip empty experiments
            if experiment_df.empty:
                print(f"Skipping empty experiment: {experiment_key}")
                continue

            # This will be the new DataFrame for this single experiment
            # with all the lagged/leaded columns
            lagged_df_single_experiment = pd.DataFrame()

            # Get the names of all signals to process (e.g., 'x', 'v', 'f_kN')
            signal_columns = experiment_df.columns

            for col_name in signal_columns:
                original_series = experiment_df[col_name]
                
                # Get the first and last values for boundary padding
                first_val = original_series.iloc[0]
                last_val = original_series.iloc[-1]

                # 1. Add Past (Minus) columns
                # e.g., for num_points=3, this loops i=3, 2, 1
                for i in range(num_points, 0, -1):
                    new_col_name = f"{col_name}_m_{i}"
                    # shift(i) moves data down, creating NaNs at the top.
                    # fill_value=first_val handles the start boundary condition.
                    lagged_df_single_experiment[new_col_name] = original_series.shift(
                        periods=i, fill_value=first_val
                    )

                # 2. Add Current value column
                lagged_df_single_experiment[col_name] = original_series

                # 3. Add Future (Plus) columns
                # e.g., for num_points=3, this loops i=1, 2, 3
                for i in range(1, num_points + 1):
                    new_col_name = f"{col_name}_p_{i}"
                    # shift(-i) moves data up, creating NaNs at the bottom.
                    # fill_value=last_val handles the end boundary condition.
                    lagged_df_single_experiment[new_col_name] = original_series.shift(
                        periods=-i, fill_value=last_val
                    )

            # Add identifier columns to know which experiment this data came from
            lagged_df_single_experiment['experiment_key'] = str(experiment_key)
            if len(experiment_key) > 0:
                lagged_df_single_experiment['experiment_name'] = experiment_key[0]
            
            # Add the fully processed DataFrame for this experiment to our list
            all_experiment_dfs.append(lagged_df_single_experiment)

        except Exception as e:
            print(f"Warning: Could not process experiment {experiment_key}. Error: {e}")
            continue

    # If no data was processed, return an empty DataFrame
    if not all_experiment_dfs:
        return pd.DataFrame()

    # Concatenate all individual experiment DataFrames into one large one
    final_df = pd.concat(all_experiment_dfs, ignore_index=True)
    
    # Optional: Re-order columns to put identifiers first
    id_cols = [col for col in final_df.columns if 'experiment_' in col]
    data_cols = [col for col in final_df.columns if col not in id_cols]
    final_df = final_df[id_cols + data_cols]

    return final_df


In [ ]:
# --- Example Usage ---

past_future_points = 2  # This will create _m_2, _m_1, x, _p_1, _p_2

# 3. Generate the DataFrame
time_series_df = create_lagged_dataframe(
    models_data=noisy_data,
    num_points=past_future_points
)

print(f"--- Generated DataFrame with num_points = {past_future_points} ---")

print("\n--- Focus on 'x' columns for first experiment ---")
cols_x = ['experiment_key'] + [col for col in time_series_df.columns if 'x' in col]
print(time_series_df[cols_x].head(4))

print("\n--- Focus on 'x' columns for second experiment ---")
print(time_series_df[cols_x].tail(3))

In [ ]:
time_series_df.columns

In [ ]:
class StopAfterOverfitting(Callback):
    """
    A custom callback designed to halt training specifically when generalization 
    performance degrades relative to training performance.

    Unlike standard EarlyStopping, this controller enforces a dual condition 
    for termination, ensuring that training is only stopped when true overfitting 
    is observed.
    
    Termination Conditions:
    1. Stagnation: The monitored metric (e.g., 'val_loss') has not improved 
       for a defined number of epochs ('patience').
    2. Divergence: The validation loss is strictly greater than the training 
       loss, indicating high variance (memorization of noise).

    Args:
        monitor (str): Quantity to be monitored (default: 'val_loss').
        patience (int): Number of epochs with no improvement after which 
                        training will be stopped.
        verbose (int): Verbosity mode.
        restore_best_weights (bool): Whether to restore model weights from 
                                     the epoch with the best value of the monitored quantity.
    """

    def __init__(self, monitor='val_loss', patience=100, verbose=1, restore_best_weights=True):
        super(StopAfterOverfitting, self).__init__()
        self.monitor = monitor
        self.patience = patience
        self.verbose = verbose
        self.restore_best_weights = restore_best_weights
        
        # Initialize internal state variables
        self.wait = 0
        self.stopped_epoch = 0
        self.best_weights = None
        self.best_loss = np.Inf

    def on_train_begin(self, logs=None):
        # Reset state at the start of training
        self.wait = 0
        self.stopped_epoch = 0
        self.best_loss = np.Inf
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        current_val_loss = logs.get(self.monitor)
        current_train_loss = logs.get('loss')

        if current_val_loss is None:
            return

        # Check for improvement in the monitored metric (Validation Loss)
        if current_val_loss < self.best_loss:
            self.best_loss = current_val_loss
            self.wait = 0
            if self.restore_best_weights:
                self.best_weights = self.model.get_weights()
        else:
            self.wait += 1

        # Evaluate the dual stopping criteria:
        # 1. Patience threshold exceeded.
        # 2. Validation loss exceeds Training loss (Overfitting/Divergence).
        if self.wait >= self.patience and current_val_loss > current_train_loss:
            self.stopped_epoch = epoch
            self.model.stop_training = True
            
            if self.restore_best_weights and self.best_weights is not None:
                if self.verbose > 0:
                    print(f"   > Overfitting detected (Val Loss > Train Loss). Restoring model weights from the best epoch.")
                self.model.set_weights(self.best_weights)

    def on_train_end(self, logs=None):
        if self.stopped_epoch > 0 and self.verbose > 0:
            print(f"Training terminated early at epoch {self.stopped_epoch + 1} due to the custom overfitting condition.")

In [ ]:
def _build_model_from_config(config: Dict[str, Any]) -> Model:
    """
    Internal helper function to build a Keras model from a configuration dictionary.

    This function creates an encoder and a mirrored decoder structure.
    The layer sizes are defined as integer multiples of the input feature dimension.
    All hidden layers use 'tanh' activation.
    """

    network_size = config['network_size']
    latent_dim = config['latent_dim']
    num_features_in = config['num_features_in']
    num_features_out = config['num_features_out']

    input_layer = Input(shape=(num_features_in,))
    
    # --- Encoder ---
    # The chain of layers 'x' will lead to the latent space
    x = input_layer 
    
    if network_size == 'Small':
        # Simple structure: In -> 2*In -> 1*In -> Latent
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        
    elif network_size == 'Medium':
        # In -> 3*In -> 2*In -> 1*In -> Latent
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(np.min([int(num_features_in * 0.5),latent_dim]), activation='tanh')(x)
        
    elif network_size == 'Large':
        # Follows the "4x, 3x..." example
        # In -> 4*In -> 3*In -> 2*In -> 1*In -> Latent
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(np.min([int(num_features_in * 0.5),latent_dim]), activation='tanh')(x)
    else:
        raise ValueError("Unknown network size. Use 'Small', 'Medium', or 'Large'.")

    # --- Latent Space ---
    # The encoder's final compression
    latent_space = Dense(latent_dim, activation='tanh')(x)

    # --- Decoder ---
    # Start the decoder chain from the latent space
    # This section mirrors the encoder's hidden layers in reverse
    x = latent_space
    
    if network_size == 'Small':
        # Latent -> 1*In -> 2*In -> Output
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
        
    elif network_size == 'Medium':
        # Latent -> 1*In -> 2*In -> 3*In -> Output
        x = Dense(np.min([int(num_features_in * 0.5),latent_dim]), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
        
    elif network_size == 'Large':
        # Latent -> 1*In -> 2*In -> 3*In -> 4*In -> Output
        x = Dense(np.min([int(num_features_in * 0.5),latent_dim]), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 1 + 5), activation='tanh')(x)
        x = Dense(int(num_features_in * 2 + 5), activation='tanh')(x)
    
    # --- Output Layer ---
    # Final layer for regression with 'linear' activation
    output_layer = Dense(num_features_out, activation='linear')(x)
    
    model = Model(input_layer, output_layer, name=f"{network_size}_{latent_dim}D")
    return model

In [ ]:
# --- CONFIGURAÇÕES DE ESTILO CIENTÍFICO ---
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman'],
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.5,
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'lines.linewidth': 2.5,
    'legend.frameon': True,
    'legend.framealpha': 0.9,
    'legend.edgecolor': 'black',
    'legend.fancybox': False,
    'figure.dpi': 150
})

FONT_CONFIG = {
    'title_size': 20,
    'axis_label_size': 18,
    'tick_label_size': 14,
    'legend_size': 14
}

COLORS = {
    'train': '#0072B2',   # Azul
    'val': '#009E73',     # Verde
    'best': '#D55E00'     # Vermelho
}

def plot_training_history(history, model_name, loss_name="Loss"):
    """
    Gera e salva o gráfico de histórico de treinamento seguindo o padrão científico.
    """
    epochs = range(1, len(history.history['loss']) + 1)
    loss_train = history.history['loss']
    loss_val = history.history['val_loss']
    
    # Encontrar melhor época
    best_epoch_index = np.argmin(loss_val)
    best_epoch_val = best_epoch_index + 1

    fig, ax = plt.subplots(figsize=(10, 6))

    # Plotagem
    ax.plot(epochs, loss_train, color=COLORS['train'], label='Training Loss', linestyle='-')
    ax.plot(epochs, loss_val, color=COLORS['val'], label='Validation Loss', linestyle='--')
    
    # Linha vertical na melhor época
    ax.axvline(x=best_epoch_val, color=COLORS['best'], linestyle='-.', linewidth=2,
               label=f'Best Epoch ({best_epoch_val})')

    # Estilização
    ax.set_title(f'Final Model Training History ({model_name})', fontsize=FONT_CONFIG['title_size'], pad=15)
    ax.set_xlabel('Epoch', fontsize=FONT_CONFIG['axis_label_size'])
    ax.set_ylabel(loss_name, fontsize=FONT_CONFIG['axis_label_size'])
    
    ax.set_yscale('log')
    ax.grid(True, which="major", linestyle='-', linewidth=0.5, alpha=0.3)
    ax.grid(True, which="minor", linestyle=':', linewidth=0.5, alpha=0.15)
    
    ax.tick_params(axis='both', labelsize=FONT_CONFIG['tick_label_size'])
    ax.legend(fontsize=FONT_CONFIG['legend_size'], loc='upper right')

    # Salvar e Fechar
    plot_filename = "final_model_training_history.png"
    plt.tight_layout()
    plt.savefig(plot_filename, bbox_inches='tight')
    plt.close(fig) # Importante para liberar memória
    print(f"Final training history plot saved to {plot_filename}")

In [ ]:
def train_final_model(
    untrained_model: Model, 
    full_dataframe: pd.DataFrame, 
    input_variables: List[str], 
    output_variables: List[str]
) -> Tuple[Model, pd.DataFrame]:
    """
    Trains a given model architecture on the full dataset using global scientific plotting standards.
    """
    
    print("\n--- Starting Final Training ---")
    print(f"Training model {untrained_model.name} on {len(full_dataframe)} samples.")

    # 1. Final Hyperparameters
    LEARNING_RATE = 0.001
    EPOCHS = 5000
    LOSS_FUNCTION = 'mean_absolute_error'
    BATCH_SIZE = 32
    
    # 2. Prepare full dataset
    X_data = full_dataframe[input_variables]
    y_data = full_dataframe[output_variables]

    # 3. Scaling (on 100% of data)
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_data)
    
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y_data)

    # 4. Create Train/Validation split
    X_train, X_val, y_train, y_val = train_test_split(
        X_scaled, y_scaled, test_size=0.1, random_state=42
    )

    print(f"Final params: Epochs={EPOCHS}, LR={LEARNING_RATE}, BatchSize={BATCH_SIZE}")
    print(f"Using {len(X_train)} samples for training, {len(X_val)} for validation.")

    # 6. Re-compile
    optimizer = Adam(learning_rate=LEARNING_RATE)
    untrained_model.compile(optimizer=optimizer, loss=LOSS_FUNCTION)
    
    # 7. Callbacks
    stop_callback = StopAfterOverfitting(
        monitor='val_loss', 
        patience=100, 
        restore_best_weights=True, 
        verbose=0
    )
    
    # 8. Fit
    history = untrained_model.fit(
        X_train, 
        y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[stop_callback],
        verbose=0
    )
    
    print("--- Final Training Complete ---")

    # --- 9. Generate and Save Plot (NEW SCIENTIFIC STANDARD) ---
    # Aqui substituímos o bloco manual pela chamada da função auxiliar
    plot_training_history(
        history=history, 
        model_name=untrained_model.name, 
        loss_name=LOSS_FUNCTION
    )

    # --- 10. Generate Y_output_model ---
    print("Generating predictions for the full dataset...")
    y_pred_scaled = untrained_model.predict(X_scaled)
    y_pred_unscaled = scaler_y.inverse_transform(y_pred_scaled)
    
    Y_output_model = pd.DataFrame(
        y_pred_unscaled, 
        columns=output_variables
    )

    # 11. Return
    return untrained_model, Y_output_model

In [ ]:


# 1. Define the input and output columns as specified
input_cols = [
    'x_m_2', 'x_m_1', 'x_p_1', 'x_p_2', 
    'v_m_2', 'v_m_1', 'v_p_1', 'v_p_2', 
    'f_kN_m_2', 'f_kN_m_1', 'f_kN_p_1', 'f_kN_p_2', 
    'e_m_2', 'e_m_1', 'e_p_1', 'e_p_2', 
    'v_approach_m_2', 'v_approach_m_1',
    'v_approach_p_1', 'v_approach_p_2'
]

output_cols = ['x', 'v', 'f_kN', 'e', 'v_approach']

# Note: 'x', 'v', etc. are in *both* lists.
# The search function will handle this by first selecting *all* columns
# and then dropping the output_cols to determine the final input_cols.
# We must ensure the dummy dataframe has all unique columns.

all_unique_cols = list(set(input_cols + output_cols))

# 3. --- PART 1: Run the Architecture Search ---
# best_model_arch, best_train_loss, best_val_loss = search_promising_architecture(
#     input_dataframe=time_series_df[input_cols],
#     min_latent_dim=2,
#     max_latent_dim=4,
#     output_variables=output_cols
# )

In [ ]:




def construct_comparative_analysis_dataframe(
    df_ground_truth: pd.DataFrame, 
    df_noisy_input: pd.DataFrame, 
    df_model_predictions: pd.DataFrame
) -> pd.DataFrame:
    """
    Synthesizes a unified data structure to facilitate comparative analysis between 
    ground truth, noisy inputs, and denoised model outputs.

    This function aligns the datasets, applies suffixing for clarity, and enforces 
    physical consistency constraints by filtering out non-physical artifacts 
    (e.g., negative displacement or force values).

    Args:
        df_ground_truth: The clean, reference dataset (Ground Truth).
        df_noisy_input: The dataset with injected noise used as model input.
        df_model_predictions: The denoised dataset generated by the model.

    Returns:
        pd.DataFrame: A consolidated DataFrame containing aligned triplets of 
                      (Truth, Noisy, Predicted) data.
    """

    # 1. Data Alignment
    # Resetting indices is crucial to ensure strict row-wise alignment during concatenation,
    # preventing mismatch errors if input DataFrames have discontinuous indices.
    df_truth_reset = df_ground_truth.reset_index(drop=True)

    # 2. Schema Standardization
    # Apply specific suffixes to distinguish between data sources in the merged view.
    
    # Rename Noisy Input columns (e.g., 'x' -> 'x_noisy')
    map_noisy = {col: f"{col}_noisy" for col in df_noisy_input.columns}
    df_noisy_renamed = df_noisy_input.reset_index(drop=True).rename(columns=map_noisy)

    # Rename Model Output columns (e.g., 'x' -> 'x_output')
    map_output = {col: f"{col}_output" for col in df_model_predictions.columns}
    df_output_renamed = df_model_predictions.reset_index(drop=True).rename(columns=map_output)

    # 3. Horizontal Concatenation
    # Merge all datasets side-by-side to create the master analysis table.
    df_analysis = pd.concat(
        [df_truth_reset, df_noisy_renamed, df_output_renamed], 
        axis=1
    )
    
    print("Enforcing physical constraints (Filtering non-physical negative values for x and f_kN)...")
    
    # 4. Physical Consistency Filtering
    # Define critical columns that must satisfy physical constraints (Value >= 0)
    # Note: 'x' denotes indentation depth and 'f_kN' denotes contact force.
    filter_cols = ['x_output', 'f_kN_output', 'x_noisy', 'f_kN_noisy']
    
    # Pre-validation: Ensure all required columns exist before filtering
    cols_exist = all(col in df_analysis.columns for col in filter_cols)
    
    if not cols_exist:
        print(f"Warning: Missing required filter columns from set {filter_cols}. Returning unfiltered DataFrame.")
        return df_analysis

    try:
        # Create a boolean mask where ALL conditions must be true
        valid_physics_mask = (
            (df_analysis['x_output'] >= 0) &
            (df_analysis['f_kN_output'] >= 0) &
            (df_analysis['x_noisy'] >= 0) &
            (df_analysis['f_kN_noisy'] >= 0)
        )
        
        df_analysis_filtered = df_analysis[valid_physics_mask].copy()
        
        rows_removed = len(df_analysis) - len(df_analysis_filtered)
        print(f"Filter applied successfully. Retained {len(df_analysis_filtered)} samples (Removed {rows_removed} non-physical artifacts).")
        
        return df_analysis_filtered

    except KeyError as e:
        print(f"Critical Error during filtering: Column {e} not found despite pre-check. Returning unfiltered data.")
        return df_analysis
        
    except Exception as e:
        print(f"Unexpected error during dataframe construction: {e}. Returning unfiltered data.")
        return df_analysis

In [ ]:

def evaluate_denoising_efficacy(
    df_analysis: pd.DataFrame, 
    stiffness_K: float
) -> pd.DataFrame:
    """
    Computes a comprehensive suite of performance metrics to evaluate the 
    efficacy of the denoising model.

    The evaluation covers three domains:
    1. Signal Fidelity: R² and MAE for kinematic variables and force.
    2. Geometric Error: Reduction in Euclidean distance in the phase space.
    3. Physical Consistency: Accuracy of force reconstruction using the 
       constitutive law (Hunt-Crossley) on denoised kinematic data.

    Hypothesis Testing (Paired t-test):
        H0: The model error is greater than or equal to the noisy input error.
        H1: The model error is significantly lower than the noisy input error (Improvement).

    Args:
        df_analysis (pd.DataFrame): DataFrame containing aligned triplets 
                                    (Clean, Noisy, Output).
        stiffness_K (float): Material stiffness parameter [N/m^1.5].

    Returns:
        pd.DataFrame: A single-row DataFrame containing all calculated metrics.
    """

    variables = ['x', 'v', 'f_kN']
    metrics = {} 
    
    # --- 1. Signal Fidelity (R² and MAE) ---
    for var in variables:
        col_clean = f"{var}_clean"
        col_noisy = f"{var}_noisy"
        col_output = f"{var}_output"
        
        y_true = df_analysis[col_clean]
        y_noisy = df_analysis[col_noisy]
        y_denoised = df_analysis[col_output]
        
        # Coefficient of Determination (R²)
        metrics[f"R2_{var}_noisy"] = r2_score(y_true, y_noisy)
        metrics[f"R2_{var}_output"] = r2_score(y_true, y_denoised)
        
        # Mean Absolute Error (MAE)
        metrics[f"MAE_{var}_noisy"] = mean_absolute_error(y_true, y_noisy)
        metrics[f"MAE_{var}_output"] = mean_absolute_error(y_true, y_denoised)

    # --- 2. Geometric Error Analysis (Euclidean Distance) ---
    # We calculate the combined error vector magnitude in the [x, v, f] space
    
    dist_noisy_sq = (
        (df_analysis['x_clean'] - df_analysis['x_noisy'])**2 +
        (df_analysis['v_clean'] - df_analysis['v_noisy'])**2 +
        (df_analysis['f_kN_clean'] - df_analysis['f_kN_noisy'])**2
    )
    dist_vector_noisy = np.sqrt(dist_noisy_sq)

    dist_output_sq = (
        (df_analysis['x_clean'] - df_analysis['x_output'])**2 +
        (df_analysis['v_clean'] - df_analysis['v_output'])**2 +
        (df_analysis['f_kN_clean'] - df_analysis['f_kN_output'])**2
    )
    dist_vector_output = np.sqrt(dist_output_sq)
    
    metrics["mean_euclidean_error_noisy"] = np.mean(dist_vector_noisy)
    metrics["std_euclidean_error_noisy"] = np.std(dist_vector_noisy)
    metrics["mean_euclidean_error_output"] = np.mean(dist_vector_output)
    metrics["std_euclidean_error_output"] = np.std(dist_vector_output)

    # --- 3. Statistical Significance Test (Euclidean Distance) ---
    # Paired T-Test (One-tailed: 'less')
    
    try:
        t_stat_dist, p_val_dist = ttest_rel(
            dist_vector_output, 
            dist_vector_noisy, 
            alternative='less',
            nan_policy='omit'
        )
        
        metrics["T_stat_distance"] = t_stat_dist
        metrics["P_value_distance"] = p_val_dist
        metrics["Hypothesis_Test_Distance"] = "Reject H0 (Significant Improvement)" if p_val_dist < 0.05 else "Fail to Reject H0"
            
    except Exception as e:
        metrics["Hypothesis_Test_Distance"] = f"Test Failed: {e}"
        metrics["T_stat_distance"] = np.nan
        metrics["P_value_distance"] = np.nan

    # --- 4. Physical Consistency Verification ---
    # We check if the denoised kinematic data (x, v) reproduces the correct Force
    # when fed back into the analytical Hunt-Crossley equation.
    
    # 

    # Helper function wrapper for apply (vectorization preferred for large data, but apply is safe here)
    def compute_force_wrapper(row, col_x, col_v):
        return hunt_crossley_force(
            delta=row[col_x],
            delta_dot=row[col_v],
            K=stiffness_K,
            e=row['e_clean'],
            v_approach=row['v_approach_clean']
        )

    # Re-calculate Force based on NOISY kinematics
    f_recalc_noisy_N = df_analysis.apply(
        lambda row: compute_force_wrapper(row, 'x_noisy', 'v_noisy'), axis=1
    )
    f_recalc_noisy_kN = f_recalc_noisy_N / 1000.0

    # Re-calculate Force based on DENOISED (Output) kinematics
    f_recalc_output_N = df_analysis.apply(
        lambda row: compute_force_wrapper(row, 'x_output', 'v_output'), axis=1
    )
    f_recalc_output_kN = f_recalc_output_N / 1000.0

    # Calculate absolute errors relative to Ground Truth Force
    abs_err_force_noisy = np.abs(df_analysis['f_kN_clean'] - f_recalc_noisy_kN)
    abs_err_force_output = np.abs(df_analysis['f_kN_clean'] - f_recalc_output_kN)

    # Metrics for Physical Consistency
    metrics["MAE_physics_consistency_noisy"] = np.mean(abs_err_force_noisy)
    metrics["MAE_physics_consistency_output"] = np.mean(abs_err_force_output)
    metrics["STD_physics_consistency_noisy"] = np.std(abs_err_force_noisy)
    metrics["STD_physics_consistency_output"] = np.std(abs_err_force_output)

    # --- 5. Statistical Significance Test (Physical Consistency) ---
    try:
        t_stat_phy, p_val_phy = ttest_rel(
            abs_err_force_output, 
            abs_err_force_noisy, 
            alternative='less',
            nan_policy='omit'
        )
        
        metrics["T_stat_physics"] = t_stat_phy
        metrics["P_value_physics"] = p_val_phy
        metrics["Hypothesis_Test_Physics"] = "Reject H0 (Physics Restored)" if p_val_phy < 0.05 else "Fail to Reject H0"
            
    except Exception as e:
        metrics["Hypothesis_Test_Physics"] = f"Test Failed: {e}"

    # Return results as a single-row DataFrame
    return pd.DataFrame(metrics, index=[0])

In [ ]:

def perform_noise_sensitivity_analysis(
    clean_experiments_dict: Dict[Any, Any],
    df_ground_truth: pd.DataFrame,
    stiffness_K: float
) -> pd.DataFrame:
    """
    Orchestrates a comprehensive sensitivity analysis loop.
    
    This function iteratively injects varying levels of Gaussian noise into the 
    clean dataset, retrains the denoising model for each noise scenario, and 
    evaluates its performance. This quantifies the model's robustness and 
    generalization capability under increasingly adverse conditions.

    Args:
        clean_experiments_dict: Dictionary containing raw, noise-free experimental data.
        df_ground_truth: The consolidated DataFrame of clean data (Target).
        stiffness_K: Material stiffness parameter [N/m^1.5] for physical verification.

    Returns:
        pd.DataFrame: A consolidated summary table containing performance metrics 
                      for every tested noise level.
    """
    
    # 1. Define Experimental Scope
    # Noise levels to test (Standard Deviation as % of signal amplitude)
    noise_levels_pct = [3, 5, 7, 10, 15]
    
    # Variables subjected to noise injection
    noise_target_vars = ['x', 'v', 'f_kN']
    
    # Container for aggregating results from each iteration
    batch_results_list = []
    
    # 2. Feature Engineering Definitions
    # 'input_features': Temporal context window (Past, Present, Future)
    input_features = [
        'x_m_2', 'x_m_1', 'x_p_1', 'x_p_2', 
        'v_m_2', 'v_m_1', 'v_p_1', 'v_p_2', 
        'f_kN_m_2', 'f_kN_m_1', 'f_kN_p_1', 'f_kN_p_2', 
        'e_m_2', 'e_m_1', 'e_p_1', 'e_p_2', 
        'v_approach_m_2', 'v_approach_m_1',
        'v_approach_p_1', 'v_approach_p_2'
    ]
    
    # 'target_features': The specific variables at t=0 to be reconstructed
    target_features = ['x', 'v', 'f_kN', 'e', 'v_approach']
    
    print(f"--- Initiating sensitivity sweep across {len(noise_levels_pct)} noise levels (Range: {min(noise_levels_pct)}% - {max(noise_levels_pct)}%) ---")
    
    # --- Main Execution Loop ---
    for i, noise_pct in enumerate(noise_levels_pct):
        print(f"\n>> Processing Scenario {i+1}/{len(noise_levels_pct)}: Noise Level = {noise_pct:.2f}%")
        
        try:
            # Phase 1: Data Synthesis (Noise Injection)
            print(f"   Phase 1/6: Injecting Gaussian noise ({noise_pct}%) into raw signals...")
            noisy_experiments_dict = add_gaussian_noise(
                models_data=clean_experiments_dict,
                noise_keys=noise_target_vars,
                noise_level_percent=noise_pct
            )
            
            # Phase 2: Feature Engineering (Lag Generation)
            print("   Phase 2/6: Constructing temporal context (Lagged DataFrame, window=2)...")
            df_lagged_features = create_lagged_dataframe(
                models_data=noisy_experiments_dict,
                num_points=2
            )
            

            # Phase 3: Model Initialization & Training
            # We re-initialize the model to avoid data leakage from previous runs
            config = {
                'network_size': 'Medium',
                'latent_dim': 3,
                'num_features_in': 20, 
                'num_features_out': 5
            }
            
            print(f"   Phase 3/6: Training Denoising Autoencoder (Config: {config['network_size']})...")
            model_instance = _build_model_from_config(config)
            
            trained_model, df_predictions = train_final_model(
                untrained_model=model_instance,
                full_dataframe=df_lagged_features,
                input_variables=input_features,
                output_variables=target_features
            )

            # Phase 4: Comparative Data Assembly
            print("   Phase 4/6: Synthesizing comparative analysis dataset (Truth vs. Noisy vs. Predicted)...")
            # Note: df_lagged_features[target_features] represents the Noisy Input at t=0
            df_analysis = construct_comparative_analysis_dataframe(
                df_ground_truth=df_ground_truth, 
                df_noisy_input=df_lagged_features[target_features], 
                df_model_predictions=df_predictions
            )
            
            if df_analysis.empty:
                print("   Warning: Analysis DataFrame is empty (likely due to strict filtering). Skipping iteration.")
                continue

            # Phase 5: Metric Calculation
            print("   Phase 5/6: Calculating performance metrics and physical consistency...")
            df_metrics_batch = evaluate_denoising_efficacy(
                df_analysis=df_analysis, 
                stiffness_K=stiffness_K
            )
            
            print(f"      > Performance Summary: \n{df_metrics_batch.to_string(index=False)}")

            # Phase 6: Result Aggregation
            print("   Phase 6/6: Archiving results.")
            df_metrics_batch['noise_level_pct'] = noise_pct
            batch_results_list.append(df_metrics_batch)

        except Exception as e:
            print(f"   FATAL ERROR at Noise Level {noise_pct}%: {e}")
            print("   Skipping to next iteration.")
            continue

    print("\n--- Sensitivity Analysis Completed ---")
    
    if not batch_results_list:
        print("No valid results were generated.")
        return pd.DataFrame()

    # Consolidate all batches into a single DataFrame
    df_final_results = pd.concat(batch_results_list, ignore_index=True)
    
    # Reorder columns to prioritize the independent variable (Noise Level)
    cols = ['noise_level_pct'] + [col for col in df_final_results.columns if col != 'noise_level_pct']
    df_final_results = df_final_results[cols]
    
    return df_final_results

In [ ]:
K = sim_params_base['K']

df_results = perform_noise_sensitivity_analysis(
            clean_experiments_dict=experiments_models,
            df_ground_truth=df_clean,
            stiffness_K=K
        )

In [ ]:
import openpyxl
df2 = df_results.copy()
with pd.ExcelWriter('output_blind_denoising_impact_experiment_new_void_central.xlsx') as writer:  
    df_results.to_excel(writer, sheet_name='Sheet_name_1')

In [ ]:


df_read  = pd.read_excel('output_blind_denoising_impact_experiment_new_void_central.xlsx')

In [ ]:
df_read.columns

In [ ]:

df_read['SNR'] = 20*np.log10 (100/ df_read['noise_level_pct'])

In [ ]:
import openpyxl
with pd.ExcelWriter('output_blind_denoising_impact_experiment_new_void_central.xlsx') as writer:  
    df_read.to_excel(writer, sheet_name='Sheet_name_1')